# ALMASim staged API walkthrough

This notebook demonstrates a Python-first workflow for:

1. querying ALMA metadata directly from TAP
2. saving the queried metadata rows as CSV
3. clean cube generation
4. in-memory interferometric simulation
5. ML shard export for DDRM training/validation


In [ ]:
from pathlib import Path

import h5py

from almasim import export_results, generate_clean_cube, simulate_observation
from almasim.services import astro
from almasim.services.astro.spectral import sample_given_redshift
from almasim.services.compute import create_backend
from almasim.services.metadata.tap import InclusionFilters, query_metadata_by_science
from almasim.services.simulation import SimulationParams

repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "almasim").exists():
    repo_root = repo_root.parent

output_dir = repo_root / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
metadata_csv = output_dir / "staged_pipeline_metadata.csv"
ml_shard_path = output_dir / "ddrm_notebook_sample.h5"


In [ ]:
include = InclusionFilters(
    science_keyword=["Galaxies"],
    band=[6],
)
metadata = query_metadata_by_science(include=include).head(25).reset_index(drop=True)
metadata.to_csv(metadata_csv, index=False)

print(f"Saved metadata CSV: {metadata_csv}")
metadata[[col for col in ["ALMA_source_name", "Band", "Freq", "member_ous_uid"] if col in metadata.columns]].head()


In [ ]:
rest_frequency, _ = astro.get_line_info(repo_root / "src" / "almasim")
sampled = sample_given_redshift(
    metadata,
    n=1,
    rest_frequency=rest_frequency,
    extended=False,
    zmax=None,
)
row = sampled.iloc[0]

params = SimulationParams.from_metadata_row(
    row,
    idx=0,
    main_dir=repo_root / "src" / "almasim",
    output_dir=output_dir,
    tng_dir=output_dir / "tng",
    galaxy_zoo_dir=output_dir / "galaxy_zoo",
    hubble_dir=output_dir / "hubble",
    project_name="ddrm_notebook_demo",
    source_type="point",
    n_pix=128,
    n_channels=32,
    save_mode="memory",
    persist=False,
    ml_dataset_path=ml_shard_path,
)

params


In [ ]:
with create_backend("sync") as backend:
    clean_stage = generate_clean_cube(
        params,
        logger=print,
        compute_backend=backend,
    )

clean_stage.model_cube.shape


In [ ]:
with create_backend("sync") as backend:
    simulation_results = simulate_observation(
        clean_stage,
        compute_backend=backend,
        robust=0.0,
    )

{
    "clean_cube": simulation_results["model_cube"].shape,
    "dirty_cube": simulation_results["dirty_cube"].shape,
    "dirty_vis": simulation_results["dirty_vis"].shape,
    "uv_mask_cube": simulation_results["uv_mask_cube"].shape,
}


In [ ]:
exported_results = export_results(
    params,
    clean_stage,
    simulation_results,
    logger=print,
)

exported_results["ml_dataset_path"]


In [ ]:
with h5py.File(exported_results["ml_dataset_path"], "r") as h5f:
    summary = {
        "clean_cube": h5f["clean_cube"].shape,
        "dirty_cube": h5f["dirty_cube"].shape,
        "dirty_vis": h5f["dirty_vis"].shape,
        "uv_mask_cube": h5f["uv_mask_cube"].shape,
        "metadata_json_length": len(h5f.attrs["metadata_json"]),
    }

summary
